In [ ]:
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_sync_playwright_browser
from langchain import hub
from langchain.agents import AgentExecutor, create_react_agent
from langchain_groq import ChatGroq

import os
from dotenv import load_dotenv
# Load environment variables from .env file
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Prompt (ReAct prompt instead of OpenAI tools)
prompt = hub.pull("hwchase17/react")


prompt.template += """

IMPORTANT RULES:
- NEVER pass JSON to tools
- ONLY pass raw values like CSS selectors
- Example:
  get_elements: div.s-result-item
"""

# Browser
sync_browser = create_sync_playwright_browser()
toolkit = PlayWrightBrowserToolkit.from_browser(sync_browser=sync_browser)
tools = toolkit.get_tools()

# Groq LLM
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0
)

# ReAct agent (IMPORTANT CHANGE)
agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

command = {
    "input": "Go to https://www.amazon.com/s?k=mouse&crid=16F015VJYSM1U&sprefix=mous%2Caps%2C455&ref=nb_sb_noss_2 and give me the title and price of the first product."
}

agent_executor.invoke(command)

ModuleNotFoundError: No module named 'langchain.prompts'